# PG2GW Log Parser
Parse pg2gw.log files to extract detections and configuration data

In [ ]:
import re
from datetime import datetime
from collections import defaultdict
import json
from pathlib import Path

In [ ]:
# Define log file path
log_path = "2025-12-19_10-50-09/pg2gw/pg2gw.log"

# Sensor IDs we're tracking
SENSOR_IDS = [
    "6000000-2000-000003-2550",
    "6000000-2000-000009-2550"
]

In [ ]:
def parse_timestamp(line):
    """
    Extract timestamp from log line.
    Format: Dec 18 14:45:44 DUSUN pg2gw[442]: ...
    """
    match = re.match(r'(\w{3}\s+\d{1,2}\s+\d{2}:\d{2}:\d{2})', line)
    if match:
        ts_str = match.group(1)
        # Add year (assuming 2025 based on log content)
        try:
            ts = datetime.strptime(f"2025 {ts_str}", "%Y %b %d %H:%M:%S")
            return ts.isoformat()
        except:
            return None
    return None


def parse_detection_translated(line):
    """
    Parse: Detection translated for sensor 6000000-2000-000003-2550: pixel(103,61) -> cm(-111,124). conf = 0.500000; box: x = 53, y = 0, w = 84, h = 76
    """
    pattern = r'Detection translated for sensor ([\w-]+): pixel\((\d+),(\d+)\) -> cm\((-?\d+),(-?\d+)\)\. conf = ([\d.]+); box: x = (\d+), y = (\d+), w = (\d+), h = (\d+)'
    match = re.search(pattern, line)
    if match:
        return {
            'type': 'detection',
            'sensor_id': match.group(1),
            'pixel_x': int(match.group(2)),
            'pixel_y': int(match.group(3)),
            'cm_x': int(match.group(4)),
            'cm_y': int(match.group(5)),
            'confidence': float(match.group(6)),
            'box': {
                'x': int(match.group(7)),
                'y': int(match.group(8)),
                'w': int(match.group(9)),
                'h': int(match.group(10))
            }
        }
    return None


def parse_filtered_confidence(line):
    """
    Parse: Detection filtered out due to low confidence: 0.394749; box: x = 179, y = 0, w = 140, h = 148
    """
    pattern = r'Detection filtered out due to low confidence: ([\d.]+); box: x = (\d+), y = (\d+), w = (\d+), h = (\d+)'
    match = re.search(pattern, line)
    if match:
        return {
            'type': 'filtered_confidence',
            'confidence': float(match.group(1)),
            'box': {
                'x': int(match.group(2)),
                'y': int(match.group(3)),
                'w': int(match.group(4)),
                'h': int(match.group(5))
            }
        }
    return None


def parse_filtered_gmm(line):
    """
    Parse: Detection filtered out by GMM: OUTLIER (x=45, y=0, w=106, h=46, conf=0.50)
    """
    pattern = r'Detection filtered out by GMM: OUTLIER \(x=(\d+), y=(\d+), w=(\d+), h=(\d+), conf=([\d.]+)\)'
    match = re.search(pattern, line)
    if match:
        return {
            'type': 'filtered_novelty',
            'box': {
                'x': int(match.group(1)),
                'y': int(match.group(2)),
                'w': int(match.group(3)),
                'h': int(match.group(4))
            },
            'confidence': float(match.group(5))
        }
    return None


def parse_set_height(line):
    """
    Parse: setHeight called: 6000000-2000-000003-2550 -> 2700 cm
    """
    pattern = r'setHeight called: ([\w-]+) -> (\d+) cm'
    match = re.search(pattern, line)
    if match:
        return {
            'sensor_id': match.group(1),
            'height_cm': int(match.group(2))
        }
    return None


def get_current_sensor_from_telemetry(lines, idx):
    """
    Look backwards from current line to find the last 'Received telemetry data' 
    to determine which sensor the filtered detection belongs to.
    """
    # Search backwards for telemetry message
    for i in range(idx - 1, max(0, idx - 10), -1):
        match = re.search(r'Received telemetry data, length: \d+, from thing ([\w-]+)', lines[i])
        if match:
            return match.group(1)
    return None

In [ ]:
def parse_log_file(log_path):
    """
    Parse the entire log file and extract detections and height settings.
    
    Returns:
        sensor_detections: dict with sensor_id -> {timestamp: [detections]}
        height_settings: dict with {timestamp: {sensor_id: height}}
    """
    # Initialize data structures
    sensor_detections = {
        sensor_id: defaultdict(list) for sensor_id in SENSOR_IDS
    }
    height_settings = {}
    
    # Read all lines for context lookup
    with open(log_path, 'r') as f:
        lines = f.readlines()
    
    current_sensor = None
    
    for idx, line in enumerate(lines):
        ts = parse_timestamp(line)
        if not ts:
            continue
        
        # Track current sensor from telemetry messages
        telemetry_match = re.search(r'Received telemetry data, length: \d+, from thing ([\w-]+)', line)
        if telemetry_match:
            current_sensor = telemetry_match.group(1)
        
        # Parse detection translated (contains sensor_id)
        detection = parse_detection_translated(line)
        if detection:
            sensor_id = detection.pop('sensor_id')
            if sensor_id in sensor_detections:
                sensor_detections[sensor_id][ts].append(detection)
            continue
        
        # Parse filtered by confidence (need to infer sensor from context)
        filtered_conf = parse_filtered_confidence(line)
        if filtered_conf:
            sensor_id = current_sensor or get_current_sensor_from_telemetry(lines, idx)
            if sensor_id and sensor_id in sensor_detections:
                sensor_detections[sensor_id][ts].append(filtered_conf)
            continue
        
        # Parse filtered by GMM/novelty (need to infer sensor from context)
        filtered_gmm = parse_filtered_gmm(line)
        if filtered_gmm:
            sensor_id = current_sensor or get_current_sensor_from_telemetry(lines, idx)
            if sensor_id and sensor_id in sensor_detections:
                sensor_detections[sensor_id][ts].append(filtered_gmm)
            continue
        
        # Parse height settings
        height_data = parse_set_height(line)
        if height_data:
            if ts not in height_settings:
                height_settings[ts] = {}
            height_settings[ts][height_data['sensor_id']] = height_data['height_cm']
            continue
    
    # Convert defaultdict to regular dict for cleaner output
    sensor_detections = {
        sensor_id: dict(detections) 
        for sensor_id, detections in sensor_detections.items()
    }
    
    return sensor_detections, height_settings

In [ ]:
# Parse the log file
print(f"Parsing log file: {log_path}")
sensor_detections, height_settings = parse_log_file(log_path)

print("\n=== Parsing Complete ===")
print(f"\nSensors found: {list(sensor_detections.keys())}")

for sensor_id in sensor_detections:
    total_timestamps = len(sensor_detections[sensor_id])
    total_detections = sum(len(dets) for dets in sensor_detections[sensor_id].values())
    
    # Count by type
    type_counts = {'detection': 0, 'filtered_confidence': 0, 'filtered_novelty': 0}
    for dets in sensor_detections[sensor_id].values():
        for det in dets:
            type_counts[det['type']] += 1
    
    print(f"\n{sensor_id}:")
    print(f"  Timestamps with detections: {total_timestamps}")
    print(f"  Total events: {total_detections}")
    print(f"    - Regular detections: {type_counts['detection']}")
    print(f"    - Filtered by confidence: {type_counts['filtered_confidence']}")
    print(f"    - Filtered by novelty (GMM): {type_counts['filtered_novelty']}")

print(f"\nHeight settings: {len(height_settings)} timestamps")

In [ ]:
# Display sample data
print("\n=== Sample Detection Data ===")

for sensor_id in SENSOR_IDS:
    if sensor_detections[sensor_id]:
        print(f"\n{sensor_id}:")
        # Get first 3 timestamps with detections
        sample_timestamps = list(sensor_detections[sensor_id].keys())[:3]
        for ts in sample_timestamps:
            print(f"\n  Timestamp: {ts}")
            for det in sensor_detections[sensor_id][ts]:
                print(f"    {det}")

In [ ]:
# Display sample height settings
print("\n=== Sample Height Settings ===")
sample_height_ts = list(height_settings.keys())[:5]
for ts in sample_height_ts:
    print(f"{ts}: {height_settings[ts]}")

In [ ]:
# Save to JSON files for later use
output_dir = Path(log_path).parent

# Save sensor detections
for sensor_id in SENSOR_IDS:
    output_file = output_dir / f"{sensor_id}_detections.json"
    with open(output_file, 'w') as f:
        json.dump(sensor_detections[sensor_id], f, indent=2)
    print(f"Saved: {output_file}")

# Save height settings
height_output = output_dir / "height_settings.json"
with open(height_output, 'w') as f:
    json.dump(height_settings, f, indent=2)
print(f"Saved: {height_output}")